# Linear Regression 

Input chính:

```text
../processed_data/model_ready/gdsc2_lgbm_style_data.npz
```

File `.npz` chứa sẵn:

```text
X_train, y_train, X_val, y_val, X_test, y_test
```


In [2]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)

## 1. Load processed data



In [3]:
DATA_PATH = Path('../processed_data/model_ready/gdsc2_lgbm_style_data.npz')
METADATA_PATH = Path('../processed_data/model_ready/artifacts/gdsc2_lgbm_style_metadata.json')
OUTPUT_DIR = Path('../models/linear_regression_processed_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

loaded = np.load(DATA_PATH)
X_train = loaded['X_train']
y_train = loaded['y_train']
X_val = loaded['X_val']
y_val = loaded['y_val']
X_test = loaded['X_test']
y_test = loaded['y_test']

with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print('Loaded processed data')
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val:  ', X_val.shape, 'y_val:  ', y_val.shape)
print('X_test: ', X_test.shape, 'y_test: ', y_test.shape)
metadata

Loaded processed data
X_train: (65239, 1477) y_train: (65239,)
X_val:   (9233, 1477) y_val:   (9233,)
X_test:  (18231, 1477) y_test:  (18231,)


{'dataset': 'GDSC2',
 'split_method': 'cold_split by Cell Line_ID',
 'random_state': 42,
 'drug_feature': 'Morgan fingerprint 1024 bits',
 'cell_feature': 'StandardScaler + PCA fitted on train cell lines only',
 'cell_pca_components': 453,
 'X_train_shape': [65239, 1477],
 'X_val_shape': [9233, 1477],
 'X_test_shape': [18231, 1477],
 'saved_npz': '../processed_data/model_ready/gdsc2_lgbm_style_data.npz'}

## 2. Evaluation helpers



In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'pearson': float(pearsonr(y_true, y_pred)[0]),
        'spearman': float(spearmanr(y_true, y_pred)[0]),
    }


def train_eval_sklearn_model(model_name, model):
    start = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - start

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    result = {
        'model': model_name,
        'fit_seconds': fit_seconds,
        'val': compute_metrics(y_val, val_pred),
        'test': compute_metrics(y_test, test_pred),
    }

    joblib.dump(model, OUTPUT_DIR / f'{model_name}.joblib')
    np.save(OUTPUT_DIR / f'{model_name}_val_pred.npy', val_pred)
    np.save(OUTPUT_DIR / f'{model_name}_test_pred.npy', test_pred)

    return result, val_pred, test_pred

## 3. Train Linear Regression baseline



In [18]:
final_model = LinearRegression()
final_model.fit(X_train, y_train)

val_pred = final_model.predict(X_val)
test_pred = final_model.predict(X_test)

result = compute_metrics(y_val, val_pred)
print('Validation Metrics:\n')
for metric, value in result.items():
    print(f'{metric}: {value:.4f}')
print('\n')
result = compute_metrics(y_test, test_pred)
print('Test Metrics:\n')
for metric, value in result.items():
    print(f'{metric}: {value:.4f}')

Validation Metrics:

rmse: 1.3797
mae: 1.0529
r2: 0.7333
pearson: 0.8585
spearman: 0.8043


Test Metrics:

rmse: 1.4255
mae: 1.0854
r2: 0.7252
pearson: 0.8518
spearman: 0.7934
